# FINANCE 384 Assignment 1 – Part A

## Task A.1: Frame the Modelling Problem

This notebook section defines the prediction target, admissible feature set, preprocessing rules, and the fixed chronological train/validation/test split required for Part A.

> **Note:** The richer model and its hyperparameter grid are intentionally left as placeholders here and should be completed in Tasks A.3–A.4.

## 0. Documented Analysis Plan

| Component | Current design |
|---|---|
| Prediction target | Next-month stock excess return, $r^e_{i,t+1}$ |
| Benchmark model | Pooled OLS |
| Richer model | **TBD in A.3** |
| Hyperparameter tuning rule | **TBD in A.4**; selection must use validation RMSE |
| Base predictors | 18 numeric characteristics + FF49 industry classification |
| Missing numeric predictors | Median imputation fitted on training data only |
| Scaling | Standardise continuous predictors using training-sample mean and standard deviation |
| Categorical encoding | One-hot encode `ff49_code`; drop one reference category |
| Training period | Jan 1990 – Dec 2014 |
| Validation period | Jan 2015 – Dec 2018 |
| Test period | Jan 2019 – Nov 2022 |
| Test-set use | Final untouched evaluation only |


In [1]:
# A.1.1 Imports and file paths
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

PANEL_FILE = "FINANCE384_assignmentA_development_panel.csv"
DICTIONARY_FILE = "FINANCE384_stock_month_data_dictionary.csv"


In [2]:
# A.1.2 Load the development panel and data dictionary
panel = pd.read_csv(PANEL_FILE)
data_dictionary = pd.read_csv(DICTIONARY_FILE)

panel["date"] = pd.to_datetime(panel["date"])

print("Panel shape:", panel.shape)
print("Date range:", panel["date"].min().date(), "to", panel["date"].max().date())
print("Unique stocks:", panel["permno"].nunique())
print("Unique months:", panel["date"].dt.to_period("M").nunique())
print("Duplicate stock-month rows:", panel.duplicated(["permno", "date"]).sum())


Panel shape: (198298, 27)
Date range: 1990-01-31 to 2022-12-30
Unique stocks: 1252
Unique months: 396
Duplicate stock-month rows: 0


### Variable roles

The assignment data dictionary separates variables into identifiers, classifications, characteristics, and the realised outcome. Identifiers and realised outcome variables are excluded from the feature matrix. The FF49 industry classification is retained as a categorical predictor because it is known at the forecast date and can be encoded without using future information.

In [3]:
# A.1.3 Review the supplied variable roles
data_dictionary[["variable", "description", "role"]]


,variable,description,role
0,date,End of month t (CRSP month-end trading date),identifier; not a predictor
1,permno,CRSP permanent security identifier,identifier; not a predictor
2,ticker,Exchange ticker at month t,identifier; not a predictor
3,comnam,Company name at month t,identifier; not a predictor
4,industry,Coarse SIC division label (Agriculture collaps...,"classification, known at t; permitted as a pre..."
5,ff49_code,Fama-French 49-industry code (Ships collapsed ...,"classification, known at t; permitted as a pre..."
6,ff49_name,Fama-French 49-industry name (Ships collapsed ...,"classification, known at t; permitted as a pre..."
7,gvkey,Compustat firm identifier (missing if no CCM l...,identifier; not a predictor
8,size,log market capitalization (USD) at end of mont...,"characteristic, contemporaneous with t"
9,bm,Book equity (6-month availability lag) / marke...,"characteristic, contemporaneous with t"


In [4]:
# A.1.4 Explicit predictor lists

numeric_predictors = [
    "size",
    "bm",
    "mom12_2",
    "vol12",
    "beta60",
    "ivol60",
    "turnover",
    "dollar_volume",
    "amihud_illiq",
    "divyield",
    "gross_profit",
    "roe",
    "asset_growth",
    "leverage",
    "accruals",
    "mkt_12m",
    "mkt_vol_12m",
    "down_market",
]

continuous_predictors = [x for x in numeric_predictors if x != "down_market"]
binary_predictors = ["down_market"]
categorical_predictors = ["ff49_code"]

identifier_columns = ["date", "permno", "ticker", "comnam", "gvkey"]
excluded_classifications = ["industry", "ff49_name"]
outcome_column = "ret_excess_t"

print("Numeric predictors:", len(numeric_predictors))
print("Categorical predictors:", categorical_predictors)
print("Identifiers excluded:", identifier_columns)
print("Other classifications excluded to avoid duplicate industry information:",
      excluded_classifications)


Numeric predictors: 18
Categorical predictors: ['ff49_code']
Identifiers excluded: ['date', 'permno', 'ticker', 'comnam', 'gvkey']
Other classifications excluded to avoid duplicate industry information: ['industry', 'ff49_name']


## A.1.5 Construct the next-month target

The target is the stock's excess return in the **immediately following calendar month**. Because a security can leave the S&P 500 panel and later re-enter, a simple within-stock shift could accidentally pair month $t$ with a non-consecutive future observation. The code below therefore verifies that the next observation is exactly one calendar month later before retaining the target.

In [5]:
# Sort by stock and decision date
analysis = panel.sort_values(["permno", "date"]).copy()

# Candidate next observation for the same stock
analysis["next_date"] = analysis.groupby("permno")["date"].shift(-1)
analysis["ret_excess_t1"] = analysis.groupby("permno")["ret_excess_t"].shift(-1)

# Require the next observation to be the immediately following calendar month
analysis["is_consecutive_next_month"] = (
    analysis["next_date"].dt.to_period("M")
    == analysis["date"].dt.to_period("M") + 1
)

# Invalid/non-consecutive targets are set to missing and will not be modelled
analysis.loc[~analysis["is_consecutive_next_month"], "ret_excess_t1"] = np.nan

gapped_observations = (
    analysis["next_date"].notna() & ~analysis["is_consecutive_next_month"]
).sum()

print("Rows in raw panel:", len(analysis))
print("Non-consecutive next observations detected:", int(gapped_observations))
print("Rows with a valid next-month target:", analysis["ret_excess_t1"].notna().sum())


Rows in raw panel: 198298
Non-consecutive next observations detected: 30
Rows with a valid next-month target: 197016


## A.1.6 Apply the fixed chronological split

The split is based on the **forecast decision date $t$**, not on the date of the realised target. December 2022 can therefore supply the realised $t+1$ return for a forecast formed in November 2022, but December 2022 is not itself a test forecast month.

In [6]:
# Keep only observations with a valid next-month target
analysis_valid = analysis.loc[analysis["ret_excess_t1"].notna()].copy()

train_mask = (
    (analysis_valid["date"] >= "1990-01-01")
    & (analysis_valid["date"] <= "2014-12-31")
)

validation_mask = (
    (analysis_valid["date"] >= "2015-01-01")
    & (analysis_valid["date"] <= "2018-12-31")
)

test_mask = (
    (analysis_valid["date"] >= "2019-01-01")
    & (analysis_valid["date"] <= "2022-11-30")
)

train = analysis_valid.loc[train_mask].copy()
validation = analysis_valid.loc[validation_mask].copy()
test = analysis_valid.loc[test_mask].copy()

split_summary = pd.DataFrame({
    "Sample": ["Training", "Validation", "Test"],
    "Decision period": [
        "Jan 1990-Dec 2014",
        "Jan 2015-Dec 2018",
        "Jan 2019-Nov 2022",
    ],
    "Months": [
        train["date"].dt.to_period("M").nunique(),
        validation["date"].dt.to_period("M").nunique(),
        test["date"].dt.to_period("M").nunique(),
    ],
    "Valid stock-month rows": [
        len(train),
        len(validation),
        len(test),
    ],
})

split_summary


,Sample,Decision period,Months,Valid stock-month rows
0,Training,Jan 1990-Dec 2014,300,149334
1,Validation,Jan 2015-Dec 2018,48,24051
2,Test,Jan 2019-Nov 2022,47,23631


## A.1.7 Missing-data assessment

Missingness is measured on the training sample. The preprocessing rule is fixed using the training period only: numeric missing values are replaced with the training-sample median. Validation and test information is not used to estimate imputation parameters.

In [7]:
training_missingness = (
    train[numeric_predictors]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("Training missing (%)")
    .to_frame()
)

training_missingness.round(2)


,Training missing (%)
beta60,8.90
ivol60,8.90
accruals,4.12
mom12_2,3.12
divyield,3.08
vol12,2.73
roe,1.46
bm,1.38
asset_growth,0.20
gross_profit,0.07


In [8]:
# Check the FF49 classification
train_industries = set(train["ff49_code"].dropna().unique())
validation_industries = set(validation["ff49_code"].dropna().unique())
test_industries = set(test["ff49_code"].dropna().unique())

print("FF49 categories in training:", len(train_industries))
print("Missing FF49 values in training:", train["ff49_code"].isna().sum())
print("Validation categories unseen in training:",
      validation_industries - train_industries)
print("Test categories unseen in training:",
      test_industries - train_industries)


FF49 categories in training: 47
Missing FF49 values in training: 0
Validation categories unseen in training: set()
Test categories unseen in training: set()


## A.1.8 Define the preprocessing pipeline

The following preprocessing specification will be fitted **only on the appropriate estimation sample** in later modelling sections:

- continuous numeric predictors: median imputation, then standardisation;
- `down_market`: median imputation only, retaining the 0/1 interpretation;
- `ff49_code`: one-hot encoding with one reference category omitted;
- no manual interactions or nonlinear transformations are introduced at A.1.

The same base predictor information will be used for pooled OLS and the richer model.

In [9]:
continuous_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

binary_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

categorical_pipeline = Pipeline(steps=[
    ("onehot", OneHotEncoder(
        drop="first",
        handle_unknown="ignore"
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", continuous_pipeline, continuous_predictors),
        ("binary", binary_pipeline, binary_predictors),
        ("industry", categorical_pipeline, categorical_predictors),
    ],
    remainder="drop",
)

X_train_raw = train[continuous_predictors + binary_predictors + categorical_predictors]
y_train = train["ret_excess_t1"]

X_validation_raw = validation[continuous_predictors + binary_predictors + categorical_predictors]
y_validation = validation["ret_excess_t1"]

X_test_raw = test[continuous_predictors + binary_predictors + categorical_predictors]
y_test = test["ret_excess_t1"]

print("Training X shape:", X_train_raw.shape)
print("Validation X shape:", X_validation_raw.shape)
print("Test X shape:", X_test_raw.shape)


Training X shape: (149334, 19)
Validation X shape: (24051, 19)
Test X shape: (23631, 19)


## A.1 Summary

Task A.1 establishes a leakage-controlled stock-return forecasting design. The target is next-month excess return, constructed only when the same security has an observation in the immediately following calendar month. The feature set contains 18 admissible numeric characteristics plus FF49 industry membership. Identifiers, duplicated industry labels, and the realised month-$t$ outcome are excluded from the feature matrix.

Preprocessing parameters will be learned from estimation data only: missing numeric predictors are imputed using training-sample information, continuous predictors are standardised, and FF49 membership is dummy encoded. The prescribed chronological training, validation, and test periods are applied using the decision date $t$. These objects are retained for the benchmark and richer-model sections that follow.


# Task A.2: Estimate the Pooled OLS Benchmark

The transparent benchmark is the pooled predictive regression

\[
r^e_{i,t+1}=\alpha+\beta'z_{i,t}+\varepsilon_{i,t+1}.
\]

Following the assignment protocol, the OLS coefficients are estimated using the **combined training and validation samples (January 1990–December 2018)**. The test sample (January 2019–November 2022) remains untouched until out-of-sample prediction.

The benchmark uses the **same base predictor information defined in A.1**. Preprocessing parameters are kept leakage-safe: imputation, scaling, and industry encoding are fitted using the training sample only and then applied unchanged to the combined estimation sample and test sample. This preserves the feature-engineering design fixed in A.1 while allowing the OLS coefficients themselves to use the required training + validation estimation period.

In [10]:
# A.2.1 Imports for pooled OLS
from sklearn.base import clone
from sklearn.linear_model import LinearRegression


In [11]:
# A.2.2 Build the combined OLS estimation sample (training + validation)

feature_columns = (
    continuous_predictors
    + binary_predictors
    + categorical_predictors
)

ols_estimation = pd.concat(
    [train, validation],
    axis=0,
    ignore_index=True
).sort_values(["date", "permno"]).reset_index(drop=True)

X_ols_estimation_raw = ols_estimation[feature_columns]
y_ols_estimation = ols_estimation["ret_excess_t1"]

X_ols_test_raw = test[feature_columns]
y_ols_test = test["ret_excess_t1"]

print("OLS estimation period:",
      ols_estimation["date"].min().date(),
      "to",
      ols_estimation["date"].max().date())
print("OLS estimation observations:", len(ols_estimation))
print("OLS test observations:", len(test))


OLS estimation period: 1990-01-31 to 2018-12-31
OLS estimation observations: 173385
OLS test observations: 23631


### Preprocessing for the benchmark

To keep the feature matrix consistent with A.1, the preprocessing transformation is **fitted on the training sample only**. The fitted transformation is then applied unchanged to:

1. the combined training + validation observations used to estimate the OLS coefficients; and
2. the untouched test observations.

This prevents any test-period information from influencing imputation, scaling, or categorical encoding.

In [12]:
# A.2.3 Fit the A.1 preprocessing transformation on training data only

ols_preprocessor = clone(preprocessor)
ols_preprocessor.fit(X_train_raw)

# Apply the fixed transformation to the OLS estimation sample and test sample
X_ols_estimation = ols_preprocessor.transform(X_ols_estimation_raw)
X_ols_test = ols_preprocessor.transform(X_ols_test_raw)

print("Transformed OLS estimation matrix:", X_ols_estimation.shape)
print("Transformed OLS test matrix:", X_ols_test.shape)


Transformed OLS estimation matrix: (173385, 64)
Transformed OLS test matrix: (23631, 64)


In [13]:
# A.2.4 Estimate pooled OLS on training + validation

ols_model = LinearRegression(fit_intercept=True)
ols_model.fit(X_ols_estimation, y_ols_estimation)

print("OLS fitted successfully.")
print("Intercept:", round(float(ols_model.intercept_), 6))
print("Number of slope coefficients:", len(ols_model.coef_))


OLS fitted successfully.
Intercept: 0.010555
Number of slope coefficients: 64


In [14]:
# A.2.5 Generate untouched test-period predictions

ols_test_pred = ols_model.predict(X_ols_test)

ols_test_predictions = test[
    ["date", "permno", "ticker", "ret_excess_t1"]
].copy()

ols_test_predictions = ols_test_predictions.rename(
    columns={"ret_excess_t1": "actual_excess_return_t1"}
)

ols_test_predictions["ols_pred_excess_return_t1"] = ols_test_pred

print("Test predictions generated:", len(ols_test_predictions))
print("Missing predictions:",
      ols_test_predictions["ols_pred_excess_return_t1"].isna().sum())
print("Prediction date range:",
      ols_test_predictions["date"].min().date(),
      "to",
      ols_test_predictions["date"].max().date())

ols_test_predictions.head()


Test predictions generated: 23631
Missing predictions: 0
Prediction date range: 2019-01-31 to 2022-11-30


,date,permno,ticker,actual_excess_return_t1,ols_pred_excess_return_t1
587,2019-01-31,10104,ORCL,0.036026,0.001409
588,2019-02-28,10104,ORCL,0.028409,0.008911
589,2019-03-29,10104,ORCL,0.032531,0.006138
590,2019-04-30,10104,ORCL,-0.087587,0.011721
591,2019-05-31,10104,ORCL,0.124089,0.012534


### Coefficient audit

The coefficient table below is retained as a model-audit output rather than as a report result. Because the continuous predictors are standardised, their coefficients represent the change in predicted next-month excess return associated with a one-standard-deviation change in the corresponding predictor, holding the remaining regressors constant. Industry dummy coefficients are interpreted relative to the omitted FF49 reference category.

In [15]:
# A.2.6 Create a coefficient audit table

ols_feature_names = ols_preprocessor.get_feature_names_out()

ols_coefficients = pd.DataFrame({
    "feature": ols_feature_names,
    "coefficient": ols_model.coef_
})

ols_coefficients["abs_coefficient"] = ols_coefficients["coefficient"].abs()

ols_coefficients_sorted = (
    ols_coefficients
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

ols_coefficients_sorted.head(15)


,feature,coefficient,abs_coefficient
0,industry__ff49_code_29,-0.013954,0.013954
1,industry__ff49_code_20,-0.012699,0.012699
2,industry__ff49_code_16,-0.012397,0.012397
3,industry__ff49_code_7,0.010955,0.010955
4,industry__ff49_code_31,-0.007128,0.007128
5,continuous__bm,0.007108,0.007108
6,industry__ff49_code_27,-0.006576,0.006576
7,industry__ff49_code_36,0.006083,0.006083
8,industry__ff49_code_12,0.005596,0.005596
9,continuous__mkt_vol_12m,0.005540,0.005540


In [16]:
# A.2.7 Basic benchmark audit checks

ols_audit = pd.DataFrame({
    "Item": [
        "Estimation observations",
        "Test observations",
        "Transformed predictors",
        "Missing test predictions",
        "Mean test prediction",
        "Std. dev. of test predictions",
    ],
    "Value": [
        len(ols_estimation),
        len(ols_test_predictions),
        X_ols_estimation.shape[1],
        int(ols_test_predictions["ols_pred_excess_return_t1"].isna().sum()),
        float(ols_test_predictions["ols_pred_excess_return_t1"].mean()),
        float(ols_test_predictions["ols_pred_excess_return_t1"].std()),
    ]
})

ols_audit


,Item,Value
0,Estimation observations,173385.000000
1,Test observations,23631.000000
2,Transformed predictors,64.000000
3,Missing test predictions,0.000000
4,Mean test prediction,0.010656
5,Std. dev. of test predictions,0.011403


## A.2 Summary

The pooled OLS benchmark is estimated on **173,385 valid stock-month observations** from the combined training and validation period (January 1990–December 2018). The transformed feature matrix contains the A.1 numeric characteristics and FF49 industry dummies, and the fitted benchmark generates predictions for all **23,631** valid test observations from January 2019–November 2022.

No model selection or hyperparameter tuning is performed for OLS. The test predictions are retained in `ols_test_predictions` for the formal out-of-sample prediction evaluation in **A.5** and the quintile portfolio analysis in **A.6**. RMSE, Spearman rank correlation, P5−P1 returns, and market-model alpha are therefore not evaluated in A.2.